## Example: Using ANYI to test hypotheses

* This notebook demonstrates how the ANotated Yeast Interactome (ANYI) can be used to test hypotheses with Python

#### Hypothesis

* Our hypothesis is that proteins that are interaction hubs in yeast tend to be essential

#### Background

* **Essential** proteins are those proteins that are indispensible for growth; in other words, if you knock out the corresponding gene, the yeast cells will not grow.
* Within the Saccharomyces Genome Database (SGD), these proteins are labeled with the phenotype "inviable". These annotations are typically derived from systematic gene deletion studies.
* An interaction **hub** within a protein-protein interaction network is a protein (node) that makes a large number of connections compared to the average protein. As the degree distribution of a protein-protein interaction network is reasonably well described with a scale-free distribution, most proteins make only a few connections and a small number of hubs make many connections. 

#### Analysis plan
* ANYI integrates SGD annotations of essential proteins in the column `essential`, which is a Boolean.
    * When `essential == True`, the corresponding `node` is essential; when `essential == False`, the `node` is not essential.
* We can define whether or not a protein is an interaction hub based on a quantile cutoff. In the below analysis, we assume that a protein in the top 10% of proteins when ranked by degree centrality is an interaction hub.
    * Degree centrality values for all nodes are available in the column `degree_centrality`; we can use this information to create a Boolean hub classification
* We can use these two binary classifications of proteins as either hubs/non-hubs and essential/non-essential to build a 2 x 2 contingency table, compute an odds ratio, and then use Fisher's Exact Test to determine its significance. This amounts to testing the hypothesis that interaction hubs tend to be essential. 

#### Step 0 - Load libraries and the database
* Execute the code cell below to load required libraries as well as the database

In [1]:
import pandas as pd
from typing import Dict, Optional
import numpy as np
from scipy.stats import fisher_exact

# this is the same .pkl file used by the ANYI Browser notebook
nodes = pd.read_pickle(f"../data/nodes.pkl")

#### Step 1 - Classify proteins as hubs and non-hubs

* We need to classify proteins as hubs and non-hubs to enable the binary classification required by the 2 x 2 contingency table
* The essentiality data is already in a convenient binary format

In [2]:
# define a threshold; 0.10 means hubs are in the top 10% by degree_centrality
cut = 0.10

# get the degree_centrality value corresponding to the quantile cutoff
threshold = nodes["degree_centrality"].quantile(1 - cut)

# get a pd.DataFrame containing only hubs
nodes["hub"] = nodes["degree_centrality"] >= threshold

* we need to define a function to enable bootstrapping 95% confidence intervals for values like the odds ratio and cell values in our 2 x 2 contingency table

In [5]:
def bootstrap_ct_fast(
    df: pd.DataFrame,
    col1: str,
    col2: str,
    n: Optional[int] = None,
    B: int = 1_000_000,
    alpha: float = 0.05,
    seed: Optional[int] = 0,
    correction: float = 0.0,   
) -> Dict[str, object]:
    
    sub = df[[col1, col2]].dropna().astype(bool)
    N = len(sub)
    if N == 0:
        raise ValueError("No rows after dropping NA.")
    if n is None:
        n = N

    # observed counts
    a = int(((sub[col1]) & (sub[col2])).sum())              # True, True
    b = int(((sub[col1]) & (~sub[col2])).sum())             # True, False
    c = int(((~sub[col1]) & (sub[col2])).sum())             # False, True
    d = int(((~sub[col1]) & (~sub[col2])).sum())            # False, False

    table = np.array([[a, b],
                      [c, d]], dtype=int)

    # "observed" odds ratio from Fisher's exact test (fine to do once)
    odds_ratio_fisher, p_value = fisher_exact(table, alternative="greater")

    # also compute the standard cross-product OR (used for bootstrap below)
    or_obs = ((a + correction) * (d + correction)) / ((b + correction) * (c + correction))
    odds1_obs = (a + correction) / (b + correction)
    odds2_obs = (c + correction) / (d + correction)

    # multinomial probabilities from empirical distribution
    p = np.array([a, b, c, d], dtype=float) / N

    rng = np.random.default_rng(seed)

    # draw all bootstrap tables at once
    boot = rng.multinomial(n, p, size=B).astype(np.int32)
    boot_a, boot_b, boot_c, boot_d = boot.T  # each is (B,)

    # vectorized odds / OR with correction
    boot_odds1 = (boot_a + correction) / (boot_b + correction)
    boot_odds2 = (boot_c + correction) / (boot_d + correction)
    boot_or    = ((boot_a + correction) * (boot_d + correction)) / ((boot_b + correction) * (boot_c + correction))

    # percentile CIs
    lo_q = alpha / 2
    hi_q = 1 - alpha / 2

    def ci(x: np.ndarray):
        return (float(np.quantile(x, lo_q)), float(np.quantile(x, hi_q)))

    a_ci = ci(boot_a); b_ci = ci(boot_b); c_ci = ci(boot_c); d_ci = ci(boot_d)
    odds1_ci = ci(boot_odds1); odds2_ci = ci(boot_odds2); or_ci = ci(boot_or)

    return {
        "a": a, "b": b, "c": c, "d": d,
        "a_lower": a_ci[0], "a_upper": a_ci[1],
        "b_lower": b_ci[0], "b_upper": b_ci[1],
        "c_lower": c_ci[0], "c_upper": c_ci[1],
        "d_lower": d_ci[0], "d_upper": d_ci[1],

        # Fisher OR / p-value for the observed table 
        "odds_ratio_fisher": float(odds_ratio_fisher),
        "p_value_fisher": float(p_value),

        # bootstrap distribution information for the cross-product OR and per-row odds
        "odds_ratio": float(or_obs), "odds_ratio_lower": or_ci[0], "odds_ratio_upper": or_ci[1],
        "row1_odds": float(odds1_obs), "row1_odds_lower": odds1_ci[0], "row1_odds_upper": odds1_ci[1],
        "row2_odds": float(odds2_obs), "row2_odds_lower": odds2_ci[0], "row2_odds_upper": odds2_ci[1],

    }

In [4]:
# bootstrapping + Fisher for both networks
results = bootstrap_ct_fast(nodes,
    col1="hub",
    col2="essential",
    n=None,
    B=1_000_000,
    alpha=0.05,
    seed=42,
    correction=0.0
)

print("="*70)
print("\nOur contingency table:\n")
print("                Essential      Non-essential")
print("Hub             {:>9d}          {:>9d}     ".format(
    results['a'], results['b']))
print("Non-hub         {:>9d}          {:>9d}     ".format(
    results['c'], results['d']), "\n")
print("="*70)
print("\nFisher's Exact Test results:\n")
print(f"  Odds ratio                     = {results['odds_ratio_fisher']:.3f}")
print(f"  95% CI                         = [{results['odds_ratio_lower']:.3f}, {results['odds_ratio_upper']:.3f}]\n")
print(f"  p-value                        = {results['p_value_fisher']:.4g}\n")
print("="*70)
print("\nOdds with bootstrapped confidence intervals:\n")
print(f"  Odds(essential and MS hub)     = {results['row1_odds']:.3f}")
print(f"  95% CI                         = [{results['row1_odds_lower']:.3f}, {results['row1_odds_upper']:.3f}]\n")
print(f"  Odds(essential and MS non-hub) = {results['row2_odds']:.3f}")
print(f"  95% CI                         = [{results['row2_odds_lower']:.3f}, {results['row2_odds_upper']:.3f}]", "\n")
print("="*70)


Our contingency table:

                Essential      Non-essential
Hub                   166                227     
Non-hub               856               2678      


Fisher's Exact Test results:

  Odds ratio                     = 2.288
  95% CI                         = [1.842, 2.832]

  p-value                        = 1.119e-13


Odds with bootstrapped confidence intervals:

  Odds(essential and MS hub)     = 0.731
  95% CI                         = [0.597, 0.892]

  Odds(essential and MS non-hub) = 0.320
  95% CI                         = [0.296, 0.345] 



* the odds that a hub protein is essential are 0.731, and the odds that a non-hub protein are essential are 0.320. The odds ratio of 2.288 (= 0.731/0.320) indicates that there is a 128.8% increase (= (2.288 - 1.000) * 100%) in the odds a hub will be essential compared to a non-hub.
* our one-sided (`alternative = "greater"` within `bootstrap_ct_fast`) *p*-value is 1.1E-13, indicating by most common significance levels that hubs are more likely to be essential than non-hubs